<a href="https://colab.research.google.com/github/Fizzah-Amir14/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fizzah-Amir14/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import duckdb
from google.colab import userdata
from huggingface_hub import hf_hub_download

hf_token = userdata.get('HF_TOKEN')

file_path = "fact_content_daily_performance/month=2026-03/data_0.parquet"
local_parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=file_path,
    repo_type="dataset",
    token=hf_token
)

con = duckdb.connect()

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

In [ ]:
# Attack: deliberately inject a label-derived column and watch the score jump

# Step 1 — rebuild a naive label the same way ML-04 did (position worse than panel median)
feature_df = con.execute(f"""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) AS avg_gsc_position,
        COALESCE(SUM(gsc_impressions), 0) AS total_gsc_impressions,
        COALESCE(SUM(gsc_clicks), 0) AS total_gsc_clicks,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
             ELSE 0 END AS ctr,
        COALESCE(SUM(ga4_pageviews), 0) AS total_ga4_pageviews,
        COALESCE(SUM(ga4_sessions), 0) AS total_ga4_sessions,
        MAX(CASE WHEN client_has_gsc THEN 1 ELSE 0 END) AS has_gsc_bool,
        MAX(CASE WHEN client_has_ga4 THEN 1 ELSE 0 END) AS has_ga4_bool,
        COUNT(*) AS days_observed
    FROM '{local_parquet_path}'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feature_df = feature_df.fillna(0)

median_pos = labeled['avg_position'].median()
labeled['decay_label'] = (labeled['avg_position'] > median_pos).astype(int)

full = feature_df.merge(labeled[['content_hash_id', 'decay_label']], on='content_hash_id')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Honest run: explicitly drop the label-source column, not just the label itself
X_clean = full.drop(columns=['content_hash_id', 'decay_label', 'avg_gsc_position'])
y = full['decay_label']

X_train, X_test, y_train, y_test = train_test_split(X_clean, y, test_size=0.2, random_state=42, stratify=y)
model_clean = LogisticRegression(max_iter=2000).fit(X_train, y_train)
auc_clean = roc_auc_score(y_test, model_clean.predict_proba(X_test)[:, 1])
print(f"Honest AUC (no leak): {auc_clean:.4f}")

# Leaked run: now re-add the label-source column back in as the deliberate trap
X_leaked = full.drop(columns=['content_hash_id', 'decay_label']).copy()  # avg_gsc_position stays in

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaked, y, test_size=0.2, random_state=42, stratify=y)
model_leaked = LogisticRegression(max_iter=2000).fit(X_train_l, y_train_l)
auc_leaked = roc_auc_score(y_test_l, model_leaked.predict_proba(X_test_l)[:, 1])
print(f"Leaked AUC (with label-source column): {auc_leaked:.4f}")

print(f"\nJump from leak: {auc_leaked - auc_clean:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest AUC (no leak): 0.7029
Leaked AUC (with label-source column): 1.0000

Jump from leak: 0.2971


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| total_gsc_impressions | Monthly impression volume from Search Console | COALESCE(...,0) — zero impressions is a real state, not missing data | Known at decision time — trailing observed count |
| total_gsc_clicks | Monthly click volume from Search Console | Same COALESCE(...,0) handling | Known at decision time |
| ctr | Engineered: clicks/impressions, guarded against divide-by-zero | Defaults to 0 when impressions = 0 | Known at decision time |
| total_ga4_pageviews | Session-level engagement, second data source | COALESCE(...,0) — content without GA4 wiring gets 0, not dropped | Known at decision time |
| total_ga4_sessions | Session count from GA4 | Same handling as above | Known at decision time |
| has_gsc_bool / has_ga4_bool | Data-coverage flags, cast boolean → int | No nulls possible — derived from a boolean source column | Known at decision time (describes instrumentation, not outcome) |
| days_observed | Row support/count per content item in the window | N/A, always populated | Known at decision time — used for data-quality weighting only |

**Note on avg_gsc_position:** originally included as a feature, but moved to excluded after the leakage hunt in Section 3 showed it is the same column the label is thresholded from. Keeping it in the feature set produced a trivial AUC of 1.0000 even in the "honest" run — it isn't a feature, it's the label in disguise.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**What I tested:** Whether `avg_gsc_position` — the column my `decay_label` is thresholded from — was leaking into the feature set.

**First pass (accidental leak):** My initial "honest" run already scored AUC = 1.0000, before I deliberately injected anything. Digging in, `avg_gsc_position` was present in `feature_df` from Section 1, and `decay_label` is built by thresholding that exact column against the panel median. The model wasn't learning a pattern — it was reading the label back through an unchanged copy of itself.

**Fix:** Dropped `avg_gsc_position` from the feature set entirely.
- Honest AUC (no leak): **0.7029**
- Leaked AUC (avg_gsc_position re-added): **1.0000**
- Jump from leak: **0.2971**

**Takeaway:** A 0.7029 AUC is a believable, reportable number for this feature set. The 1.0000 score is not a good model — it's a data-contract violation. I keep 0.7029 and exclude avg_gsc_position from the feature vector going forward, same as the pos_last30 exclusion in ML-04's contract.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Why |
|---|---|
| avg_gsc_position (label source) | Used to construct decay_label — including it as a feature is circular, confirmed by the leakage hunt in Section 3 (AUC 0.7029 → 1.0000 when re-added) |
| imp_last30, clk_last30 | Same-window as label construction; risk of encoding position-linked signal |
| imp_prev30 | Observed near-zero across the dataset in prior EDA — no usable signal |
| client_hash_id, content_hash_id (as model input) | Identifiers used for joins/grouping only, not predictive features |
| Raw report_date | Would let the model learn March-2026-specific calendar effects that won't generalize to the sealed June test month |
| GA4-only rows with no GSC data | Filtered out via gsc_data_available IS TRUE, keeping the feature space consistent with what's available at scoring time |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.